In [11]:
!mkdir -p src

In [12]:
import os

os.makedirs("src", exist_ok=True)
os.makedirs("figures", exist_ok=True)
os.makedirs("results", exist_ok=True)


In [13]:
from src.clustering_analysis import *
import pandas as pd


In [27]:
from google.colab import files

files.download("figures/digits_kmeans_pca_clusters.png")
files.download("figures/digits_kmeans_centroids.png")
files.download("figures/digits_hierarchical_dendrogram.png")
files.download("figures/wine_hierarchical_dendrogram.png")
files.download("results/kmeans_model_selection.csv")
files.download("results/clustering_comparison.csv")
files.download("results/clustering_stability.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [14]:
X_digits, y_digits = prepare_dataset("digits")
X_wine, y_wine = prepare_dataset("wine")


In [15]:
#Task 1
df_raw = run_kmeans_experiment(X_digits, "raw")
df_std = run_kmeans_experiment(X_digits, "standardized")
df_pca = run_kmeans_experiment(X_digits, "pca_95")

df_all = pd.concat([df_raw, df_std, df_pca], ignore_index=True)
df_all.to_csv("results/kmeans_model_selection.csv", index=False)
df_all.head()


,preprocessing,k,inertia_mean,inertia_std,silhouette_mean,silhouette_std,ch_mean,db_mean
0,raw,2,1.914620e+06,0.000000,0.118328,1.387779e-17,229.165950,2.528891
1,raw,3,1.730182e+06,1.866667,0.126493,8.592419e-06,222.347348,2.438443
2,raw,4,1.610139e+06,1075.193716,0.127072,2.157535e-03,203.753163,2.235758
3,raw,5,1.497864e+06,476.163824,0.137714,8.328543e-04,197.758200,2.064755
4,raw,6,1.404974e+06,9.912876,0.151437,9.783150e-05,192.254487,1.986863


In [16]:
#Task 1.2
best_k = select_best_k(df_all, "standardized")
best_k


12

In [17]:
#Task 1.3
def plot_pca_clusters(X, k, filename):
    # Standardize
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # PCA to 95% variance (for clustering)
    pca_full = PCA(n_components=0.95)
    X_pca_full = pca_full.fit_transform(X_scaled)

    # Run K-Means on PCA-reduced data
    km = KMeans(n_clusters=k, n_init=20, max_iter=300, random_state=RANDOM_SEED)
    labels = km.fit_predict(X_pca_full)

    # PCA to 2 components (for visualization)
    pca_2 = PCA(n_components=2)
    X_2d = pca_2.fit_transform(X_pca_full)  # <-- FIXED
    centroids_2d = pca_2.transform(km.cluster_centers_)  # <-- FIXED

    var_ratio = pca_2.explained_variance_ratio_

    plt.figure(figsize=(8, 6))
    scatter = plt.scatter(X_2d[:, 0], X_2d[:, 1], c=labels, cmap="tab10", s=20)
    plt.scatter(centroids_2d[:, 0], centroids_2d[:, 1], c="black", s=80, marker="X")
    plt.xlabel(f"PC1 ({var_ratio[0]*100:.1f}% variance)")
    plt.ylabel(f"PC2 ({var_ratio[1]*100:.1f}% variance)")
    plt.title(f"K-Means Clusters (k={k}) in PCA Space")
    plt.colorbar(scatter)
    plt.tight_layout()
    plt.savefig(filename, dpi=300)
    plt.close()

    return labels




In [18]:
labels_digits = plot_pca_clusters(
    X_digits,
    best_k,
    "figures/digits_kmeans_pca_clusters.png"
)

In [19]:
km = KMeans(
    n_clusters=best_k,
    n_init=20,
    max_iter=300,
    random_state=42
)

labels_digits = km.fit_predict(X_digits)

In [20]:
from sklearn.metrics import (
    homogeneity_score,
    completeness_score,
    v_measure_score,
    adjusted_rand_score,
    normalized_mutual_info_score,
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score
)


In [21]:
#Task 1.4
metrics_digits = {
    "silhouette": silhouette_score(X_digits, labels_digits),
    "calinski_harabasz": calinski_harabasz_score(X_digits, labels_digits),
    "davies_bouldin": davies_bouldin_score(X_digits, labels_digits),
    "ari": adjusted_rand_score(y_digits, labels_digits),
    "nmi": normalized_mutual_info_score(y_digits, labels_digits),
    "homogeneity": homogeneity_score(y_digits, labels_digits),
    "completeness": completeness_score(y_digits, labels_digits),
    "v_measure": v_measure_score(y_digits, labels_digits),
}

metrics_digits


{'silhouette': np.float64(0.18278333752373876),
 'calinski_harabasz': np.float64(156.5240078363453),
 'davies_bouldin': np.float64(1.8883851342581952),
 'ari': 0.7058636655747265,
 'nmi': np.float64(0.7708914050485531),
 'homogeneity': np.float64(0.7963189052532715),
 'completeness': np.float64(0.7470375235718673),
 'v_measure': np.float64(0.7708914050485534)}

In [22]:
#Task 1.5
plot_centroids(
    X_digits,
    10,
    "figures/digits_kmeans_centroids.png"
)


In [23]:
#Task 2
plot_dendrogram(
    X_digits,
    "Digits",
    "complete",
    "figures/digits_hierarchical_dendrogram.png",
    truncate=True
)



In [24]:
plot_dendrogram(
    X_wine,
    "Wine",
    "ward",
    "figures/wine_hierarchical_dendrogram.png",
    truncate=False
)


In [25]:
#Task 3
rows = []

for name, X, y in [
    ("digits", X_digits, y_digits),
    ("wine", X_wine, y_wine)
]:
    # K-Means
    km = KMeans(n_clusters=best_k, n_init=20, max_iter=300, random_state=RANDOM_SEED)
    labels_km = km.fit_predict(X)

    rows.append({
        "dataset": name,
        "algorithm": "kmeans",
        "silhouette": silhouette_score(X, labels_km),
        "davies_bouldin": davies_bouldin_score(X, labels_km),
        "ari": adjusted_rand_score(y, labels_km),
        "nmi": normalized_mutual_info_score(y, labels_km),
    })

    # Hierarchical
    hc = AgglomerativeClustering(n_clusters=best_k, linkage="ward")
    labels_hc = hc.fit_predict(X)

    rows.append({
        "dataset": name,
        "algorithm": "hierarchical_ward",
        "silhouette": silhouette_score(X, labels_hc),
        "davies_bouldin": davies_bouldin_score(X, labels_hc),
        "ari": adjusted_rand_score(y, labels_hc),
        "nmi": normalized_mutual_info_score(y, labels_hc),
    })

    # GMM
    labels_gmm = run_gmm(X, best_k)

    rows.append({
        "dataset": name,
        "algorithm": "gmm",
        "silhouette": silhouette_score(X, labels_gmm),
        "davies_bouldin": davies_bouldin_score(X, labels_gmm),
        "ari": adjusted_rand_score(y, labels_gmm),
        "nmi": normalized_mutual_info_score(y, labels_gmm),
    })

df_comp = pd.DataFrame(rows)
df_comp.to_csv("results/clustering_comparison.csv", index=False)
df_comp


,dataset,algorithm,silhouette,davies_bouldin,ari,nmi
0,digits,kmeans,0.182783,1.888385,0.705864,0.770891
1,digits,hierarchical_ward,0.173623,1.927043,0.812886,0.868549
2,digits,gmm,0.160229,1.843086,0.592471,0.703547
3,wine,kmeans,0.516426,0.496783,0.137590,0.324927
4,wine,hierarchical_ward,0.495977,0.500600,0.140988,0.325786
5,wine,gmm,-0.207234,9.351287,0.336845,0.568319


In [26]:
#Task 4
seeds = list(range(10))

df_stab = stability_kmeans(X_digits, best_k, seeds)
df_stab.to_csv("results/clustering_stability.csv", index=False)
df_stab


,mean_inertia,std_inertia,mean_silhouette,std_silhouette,mean_pairwise_ari,std_pairwise_ari
0,1.099537e+06,1260.297256,0.183101,0.000465,0.952946,0.044752
